In [ ]:
!python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [6]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['pennylaneai_pennylane']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)
print("Cell done")

Cell done


In [7]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)
print("Cell done")

 19%|█████████████████▏                                                                       | 2455/12732 [41:48<2:55:01,  1.02s/it]


KeyboardInterrupt: 

In [8]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
print(v)
print("Cell done")

{'commit': '0000da65ea2adec04d1be9b3c810b4792e26fcc6', 'tree': '7fce355ab082d12348e6356c2f1ae3edce943a99', 'parent': ['2245f89cf00118d2aabe8633c6b946a492566fec'], 'author': 'Antal Szava <antalszava@gmail.com>', 'author_time': 1647619598, 'author_tz': '-0400', 'committer': 'Antal Szava <antalszava@gmail.com>', 'committer_time': 1647619598, 'committer_tz': '-0400', 'message': 'add indexing and len example\n'}
Cell done


In [9]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)
print("Cell done")

Cell done


In [10]:
# check if it has the right content
dfinf.head(1)
print("Cell done")

Cell done


In [11]:
#mode a means append, so you have all your projects in the same file
yournetid='spastor'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)
print("Cell done")

Cell done


# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [ ]:
Repository: nlesc_mcfly	https://github.com/NLeSC/mcfly
GitHub: 366 stars, 83 forks, last commit July 28, 2025
WOC: 14 authors, 936 commits, (could not find on WOC Explore so I used GitHub)

Repository:	cjhopp_scripts	https://github.com/cjhopp/scripts
GitHub: 9 stars, 3 forks, last commit Aug 25, 2026
WOC: 3 author, 1395 commits,

Repository:	synthetichealth_module-builder	https://github.com/synthetichealth/module-builder
GitHub: 49 stars, 29 forks, last commit Sept 25, 2026
WOC: 33 authors, 796 commits, 

Repository:	pennylaneai_pennylane	https://github.com/PennyLaneAI/pennylane
GitHub: 3500 stars, 869 forks, last commit Sept 25, 2026
WOC: 318 authors, 68389 commits, 

Repository:	jatinchowdhury18_analogtapemodel	https://github.com/jatinchowdhury18/AnalogTapeModel
GitHub: 1400 stars, 76 forks, last commit Nov 5, 2023
WOC: 20 authors, 1230 commits, 

Repository:	paddlepaddle_deepspeech	https://github.com/PaddlePaddle/DeepSpeech
GitHub: 12700 stars, 2000 forks, last commit June 12, 2026
WOC: 1 author, 1 commit, 

Repository:	fraserw_pymop	https://github.com/fraserw/PyMOP
GitHub: 23 stars, 18 forks, last commit, June 4, 2025
WOC: 8 authors, 384 commits, 

Repository:	henryseg_veering	https://github.com/henryseg/Veering
GitHub: 10 stars, 3 forks, last commit Sept 24, 2026
WOC: 4 authors, 794 commits, (could not find on WOC Explore so I used GitHub)

Repository:	tanghaibao_goatools	https://github.com/tanghaibao/goatools
GitHub: 908 stars, 212 forks, last commit Aug 21, 2026
WOC: 54 authors, 1839 commits, 

Repository:	helske_bssm	https://github.com/helske/bssm
GitHub: 49 stars, 14 forks, last commit Sept 24, 2026
WOC: 6 authors, 894 commits, 

